In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Setup

In [ ]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")

In [ ]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.auth.add_app_token(token=dagshub_token)
dagshub.init(repo_owner='ngval22', repo_name='house_prices', mlflow=True)
mlflow.get_tracking_uri()

# Data Analysis

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")


In [ ]:
print(df.dtypes)
print('---------------------------------------------')
print(df.shape)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df['SalePrice'], kde=True, color='blue')
plt.title('Distribution of SalePrice')
plt.show()

print(f"SalePrice mean is: ", df['SalePrice'].mean())

In [ ]:
from sklearn.model_selection import train_test_split

X=df.drop(columns=['SalePrice'])
y=df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print(f"Train X: {X_train.shape}")
print(f"Train y: {y_train.shape}")
print(f"Test X:  {X_test.shape}")
print(f"Test y:  {y_test.shape}")

In [ ]:
X_train.info()

In [ ]:
X_train.describe()

In [ ]:
na_percentages = X_train.isna().mean()

In [ ]:
top_na_columns=na_percentages[na_percentages > 0].sort_values(ascending=False)
top_na_columns

**აქედან შეგვიძლია დავასკვნათ რომ Bsmt... რადგან ერთნაირი nan პროცენტულობა აქვს, სავარაუდოდ ნიშნავს რომ უბრალოდ სარდაფი არ აქვს შესაბამის საცხოვრებელს, იგივე შეიძლება ითქვას გარაჟზე, Fence, Alley, PoolQC სვეტებიც სავარაუდოდ ღობის, აუზის არსებობა არარსებობაზეა და არა იმაზე რომ მონაცემი ვერ მოიძებნა, ამ სვეტების დადროპვა არ იქნება გონივრული რადგან მნიშვნელოვნად კორელირებს ფასთან**

In [ ]:
import matplotlib.pyplot as plt

colors = [
    'firebrick' if val > 0.7 else 'sandybrown' if val > 0.5 else 'skyblue' 
    for val in top_na_columns
]
plt.figure(figsize=(10, 6))
top_na_columns.plot(kind='bar', color=colors)
plt.title('Proportion of Missing Values by Column (showing only > 0)')
plt.ylabel('Fraction of Missing Values')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
numerical_cols = [col for col in X_train.columns if X_train[col].dtype != 'object']

print(f"Categorical ({len(categorical_cols)}): \n {categorical_cols} \n")
print(f"Numerical ({len(numerical_cols)}): \n {numerical_cols}")

In [ ]:
import matplotlib.pyplot as plt

X_train[numerical_cols].hist(
    bins=20, 
    figsize=(30, 100),
    layout=(20, 3),    
    grid=False         
)

plt.show()

In [ ]:
train_corr = X_train[numerical_cols].join(y_train).corr()
# top_features = train_corr['SalePrice'].sort_values(ascending=False).index

top_features = train_corr['SalePrice'].abs().sort_values(ascending=False)

top_features

In [ ]:

weak_features = train_corr.index[abs(train_corr['SalePrice']) <= 0.05]

print(f"weak features:\n", len(weak_features), weak_features)


**target-თან სუსტი კორელაციის გამო ამ სვეტებს დავდროპავ**

In [ ]:
X_train['OverallQual']

# Feature Engineering

In [ ]:
# leaky / low-signal columns
drop_cols = ['Id', 'BsmtFinSF2', 'LowQualFinSF', 'MiscVal', 'MiscFeature']

# fill N/A with median of that column (numerical)
fill_median_cols = ['LotFrontage', 'GarageYrBlt']

# fill N/A with 0 (already numerical)
fill_0_cols = ['MasVnrArea']

# Below starts categoricals

# n/a needs to be filled with the most frequent for the column and then one hot encoding done
fill_with_frequent_ohe = ['Electrical']

# needs filling with 'none' and one hot encoding
fill_with_none_ohe = ['Alley', 'Fence', 'GarageType', 'MasVnrType']

# already filled, needs one hot encoding
ohe_cols = [
    'Street', 'LotShape', 'LandContour',  'LotConfig', 
    'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
    'RoofStyle', 'RoofMatl', 
    'Foundation', 'Heating', 'CentralAir', 'Functional', 'PavedDrive', 
    'MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd',
    'SaleType', 'SaleCondition',
]

# needs filling with none and ordered encoding, different scales
fill_none_ordered_enc_cols = [
    'PoolQC', 
    'BsmtQual', 'BsmtExposure', 'BsmtCond', 
    'GarageQual', 'GarageCond', 'GarageFinish',
    'FireplaceQu',
     'BsmtFinType1', 'BsmtFinType2',
]

# ready to encode ordered way
ordered_enc_cols = ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual']

In [ ]:
QUALITY_SCALE = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']          
EXPOSURE_SCALE = ['None', 'No', 'Mn', 'Av', 'Gd']                 
FINISH_SCALE = ['None', 'Unf', 'RFn', 'Fin']
BSMTFIN_SCALE = ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
 
ORDER_MAPPINGS = {

    'PoolQC': QUALITY_SCALE,
    'BsmtQual': QUALITY_SCALE,
    'BsmtCond': QUALITY_SCALE,
    'GarageQual': QUALITY_SCALE,
    'GarageCond': QUALITY_SCALE,
    'FireplaceQu': QUALITY_SCALE,
    'BsmtExposure': EXPOSURE_SCALE,
    'GarageFinish': FINISH_SCALE,
    'BsmtFinType1': BSMTFIN_SCALE,
    'BsmtFinType2': BSMTFIN_SCALE,

    'ExterQual': QUALITY_SCALE,
    'ExterCond': QUALITY_SCALE,
    'HeatingQC': QUALITY_SCALE,
    'KitchenQual':  QUALITY_SCALE,
}

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
 
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        return X.drop(columns=[c for c in self.cols if c in X.columns], errors='ignore')
 
# fill N/As with string 'None'
class FillNoneTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
 
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            if col in X.columns:
                X[col] = X[col].fillna('None')
        return X

# apply SimpleImputer to specific columns
class _ColumnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, cols, strategy='median', fill_value=None):
        self.cols = cols
        self.strategy = strategy
        self.fill_value = fill_value
 
    def fit(self, X, y=None):
        self.present_ = [c for c in self.cols if c in X.columns]
        kw = {'strategy': self.strategy}
        if self.strategy == 'constant':
            kw['fill_value'] = self.fill_value
        self.imputer_ = SimpleImputer(**kw)
        self.imputer_.fit(X[self.present_])
        return self
 
    def transform(self, X):
        X = X.copy()
        X[self.present_] = self.imputer_.transform(X[self.present_])
        return X


In [ ]:
 
# encode while preserving quality order hierarchy
class OrdinalEncoderDF(BaseEstimator, TransformerMixin):
    def __init__(self, mappings: dict):
        self.mappings = mappings  # {col: [ordered categories]}
 
    def fit(self, X, y=None):
        self.encoders_ = {}
        for col, cats in self.mappings.items():
            if col in X.columns:
                enc = OrdinalEncoder(
                    categories=[cats],
                    handle_unknown='use_encoded_value',
                    unknown_value=-1,
                    dtype=np.float64,
                )
                enc.fit(X[[col]])
                self.encoders_[col] = enc
        return self
 
    def transform(self, X):
        X = X.copy()
        for col, enc in self.encoders_.items():
            if col in X.columns:
                X[col] = enc.transform(X[[col]])
        return X
 
# one hot encoder
class OneHotEncoderDF(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
 
    def fit(self, X, y=None):
        present = [c for c in self.cols if c in X.columns]
        self.present_ = present
        self.enc_ = OneHotEncoder(
            drop='first',
            sparse_output=False,
            handle_unknown='ignore',
            dtype=np.float64,
        )
        self.enc_.fit(X[present])
        self.feature_names_ = self.enc_.get_feature_names_out(present)
        return self
 
    def transform(self, X):
        X = X.copy()
        ohe_array = self.enc_.transform(X[self.present_])
        ohe_df = pd.DataFrame(ohe_array, columns=self.feature_names_, index=X.index)
        X = X.drop(columns=self.present_)
        X = pd.concat([X, ohe_df], axis=1)
        return X

In [ ]:
class TotalSFAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalSF'] = X['TotalBsmtSF'] + X['1stFlrSF'] + X['2ndFlrSF']
        X.drop(columns=['TotalBsmtSF', '1stFlrSF', '2ndFlrSF'], inplace=True)
        return X

class TotalBathroomsAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalBathrooms'] = X['FullBath'] + X['HalfBath']*0.5 + X['BsmtFullBath'] + X['BsmtHalfBath']*0.5
        X.drop(columns=['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath'], inplace=True)
        return X

class TotalPorchSFAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalPorchSF'] = X['OpenPorchSF'] + X['EnclosedPorch'] + X['3SsnPorch'] + X['ScreenPorch']
        X.drop(columns=['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch'], inplace=True)
        return X

# Feature Selection

# Full Pipeline

In [ ]:
all_ordinal_cols = fill_none_ordered_enc_cols + ordered_enc_cols
ordinal_mappings = {c: ORDINAL_MAPPINGS[c] for c in all_ordinal_cols if c in ORDINAL_MAPPINGS}
 
preprocessing_pipeline_0 = Pipeline([
    ('drop_cols', DropColumns(cols=drop_cols)),
    ('add_total_sf', TotalSFAdder()),
    ('add_total_bathrooms', TotalBathroomsAdder()),
    ('add_total_porch', TotalPorchSFAdder()),
    ('fill_none_ordered', FillNoneTransformer(cols=fill_none_ordered_enc_cols)),
    ('fill_none_ohe', FillNoneTransformer(cols=fill_with_none_ohe)),
    ('fill_median', _ColumnImputer(cols=fill_median_cols, strategy='median')),
    ('fill_zero', _ColumnImputer(cols=fill_0_cols, strategy='constant', fill_value=0)),
    ('fill_frequent', _ColumnImputer(cols=fill_with_frequent_ohe, strategy='most_frequent')),
    ('ordinal_encode', OrdinalEncoderDF(mappings=ordinal_mappings)),
    ('ohe_none_cols', OneHotEncoderDF(cols=fill_with_none_ohe)),
    ('ohe_freq_cols', OneHotEncoderDF(cols=fill_with_frequent_ohe)),
    ('ohe_ready_cols', OneHotEncoderDF(cols=ohe_cols)),
])

# Training